# Comprehensive Comparison of Uncertainty Estimation and Calibration Methods for Object Detection

**Complete Evaluation of Detection Performance, Calibration Quality, and Risk-Coverage Trade-offs**

## Overview

This notebook performs a comprehensive comparison of uncertainty estimation and calibration methods for object detection using Grounding DINO on the BDD100K dataset.

**Objective**: Compare 6 methods side-by-side across detection, calibration, and risk-coverage metrics.

**Methods Evaluated**:
1. Baseline (no uncertainty, no calibration)
2. Baseline + Temperature Scaling (TS)
3. MC-Dropout K=5
4. MC-Dropout K=5 + TS
5. Decoder Layer Variance (single-pass)
6. Decoder Layer Variance + TS

**Dataset Splits**:
- `val_calib`: Calibration set for optimizing temperatures
- `val_eval`: Evaluation set for final performance assessment

**Metrics**:
- **Detection**: mAP@[0.5:0.95], AP50, AP75, per-class AP
- **Calibration**: Negative Log-Likelihood (NLL), Brier Score, Expected Calibration Error (ECE), Reliability Diagrams
- **Risk-Coverage**: Risk-coverage curves and Area Under Curve (AUC)

## Optimization Strategy

This notebook is optimized to reuse results from previous phases:
- ✅ **Phase 2 (Baseline)**: Loads predictions from `fase 2/outputs/baseline/preds_raw.json`
- ✅ **Phase 3 (MC-Dropout)**: Loads predictions from `fase 3/outputs/mc_dropout/preds_mc_aggregated.json`
- ✅ **Phase 4 (Temperature Scaling)**: Loads optimized temperatures from `fase 4/outputs/temperature_scaling/temperature.json`

**Advantages**:
- 🚀 Reduces execution time from ~2 hours to ~15 minutes
- 💾 Avoids recalculating expensive predictions (especially MC-Dropout with K=5)
- ♻️ Ensures consistency with previous phase results

**Operation Mode**:
- If files exist → Loads and reuses them
- If files don't exist → Performs full inference (fallback)

## 1. Setup and Imports

In [ ]:
import os
import sys
import json
import yaml
import time
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import torchvision
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = Path('outputs/comparison')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'categories': ['person', 'rider', 'car', 'truck', 'bus', 'train', 'motorcycle', 'bicycle', 'traffic light', 'traffic sign'],
    'iou_matching': 0.5,
    'conf_threshold': 0.25,
    'nms_threshold': 0.65,
    'K_mc': 5,
    'n_bins': 10
}

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed(CONFIG['seed'])

with open(OUTPUT_DIR / 'config.yaml', 'w') as f:
    yaml.dump(CONFIG, f)

print(f"Device: {CONFIG['device']}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Configuration saved")

## 1.1 Load Previous Phase Results (Optimization)

In [ ]:
FASE2_BASELINE = BASE_DIR / 'fase 2' / 'outputs' / 'baseline' / 'preds_raw.json'
FASE3_MC_DROPOUT_PARQUET = BASE_DIR / 'fase 3' / 'outputs' / 'mc_dropout' / 'mc_stats_labeled.parquet'
FASE3_MC_DROPOUT_JSON = BASE_DIR / 'fase 3' / 'outputs' / 'mc_dropout' / 'preds_mc_aggregated.json'
FASE4_TEMPERATURE = BASE_DIR / 'fase 4' / 'outputs' / 'temperature_scaling' / 'temperature.json'
FASE4_CALIB_DATA = BASE_DIR / 'fase 4' / 'outputs' / 'temperature_scaling' / 'calib_detections.csv'

cached_predictions = {
    'baseline': None,
    'mc_dropout': None,
    'temperatures': None
}

if FASE2_BASELINE.exists():
    print(f"✅ Loading Baseline predictions from Phase 2...")
    with open(FASE2_BASELINE, 'r') as f:
        cached_predictions['baseline'] = json.load(f)
    print(f"   → {len(cached_predictions['baseline'])} predictions loaded")
else:
    print(f"⚠️  {FASE2_BASELINE} not found, will run full inference")

if FASE3_MC_DROPOUT_PARQUET.exists():
    print(f"✅ Loading MC-Dropout predictions from Phase 3 (with uncertainty)...")
    mc_df = pd.read_parquet(FASE3_MC_DROPOUT_PARQUET)
    
    cached_predictions['mc_dropout'] = []
    for _, row in mc_df.iterrows():
        bbox = row['bbox']
        if isinstance(bbox, (list, np.ndarray)) and len(bbox) == 4:
            if bbox[2] > bbox[0] and bbox[3] > bbox[1]:
                bbox_xywh = [bbox[0], bbox[1], bbox[2]-bbox[0], bbox[3]-bbox[1]]
            else:
                bbox_xywh = bbox
        else:
            bbox_xywh = bbox
            
        cached_predictions['mc_dropout'].append({
            'image_id': int(row['image_id']),
            'category_id': int(row['category_id']) + 1,
            'bbox': bbox_xywh,
            'score': float(row['score_mean']),
            'uncertainty': float(row['uncertainty'])
        })
    print(f"   → {len(cached_predictions['mc_dropout'])} predictions loaded (with uncertainty)")
elif FASE3_MC_DROPOUT_JSON.exists():
    print(f"⚠️  Loading MC-Dropout from JSON (WITHOUT uncertainty)...")
    with open(FASE3_MC_DROPOUT_JSON, 'r') as f:
        cached_predictions['mc_dropout'] = json.load(f)
    print(f"   → {len(cached_predictions['mc_dropout'])} predictions loaded")
    print(f"   ⚠️  WARNING: This file does NOT contain uncertainty, will be set to 0.0")
else:
    print(f"⚠️  {FASE3_MC_DROPOUT_PARQUET} not found, will run full inference")

if FASE4_TEMPERATURE.exists():
    print(f"✅ Loading optimized temperatures from Phase 4...")
    with open(FASE4_TEMPERATURE, 'r') as f:
        cached_predictions['temperatures'] = json.load(f)
    print(f"   → Baseline temperature: {cached_predictions['temperatures'].get('optimal_temperature', 'N/A')}")
else:
    print(f"⚠️  {FASE4_TEMPERATURE} not found, will calculate temperatures")

print(f"\n{'='*60}")
print(f"OPTIMIZATION SUMMARY:")
print(f"{'='*60}")
print(f"Baseline available:      {'✅ YES' if cached_predictions['baseline'] else '❌ NO (will calculate)'}")
print(f"MC-Dropout available:    {'✅ YES' if cached_predictions['mc_dropout'] else '❌ NO (will calculate)'}")
print(f"Temperatures available:  {'✅ YES' if cached_predictions['temperatures'] else '❌ NO (will calculate)'}")
print(f"{'='*60}\n")

In [ ]:
def convert_baseline_predictions(baseline_data, image_filename_to_id):
    """Converts Phase 2 baseline predictions to expected format."""
    converted = {}
    for pred in baseline_data:
        img_id = pred.get('image_id')
        if img_id not in converted:
            converted[img_id] = []
        
        bbox = pred['bbox']
        bbox_xyxy = [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
        score = pred['score']
        score_clipped = np.clip(score, 1e-7, 1 - 1e-7)
        logit = np.log(score_clipped / (1 - score_clipped))
        
        converted[img_id].append({
            'bbox': bbox_xyxy,
            'score': score_clipped,
            'logit': logit,
            'category_id': pred['category_id'],
            'uncertainty': pred.get('uncertainty', 0.0)
        })
    
    return converted

def convert_mc_predictions(mc_data, image_filename_to_id):
    """Converts Phase 3 MC-Dropout predictions to expected format.
    Handles both [x,y,w,h] and [x1,y1,x2,y2] bbox formats."""
    converted = {}
    for pred in mc_data:
        img_id = pred.get('image_id')
        if img_id not in converted:
            converted[img_id] = []
        
        bbox = pred['bbox']
        
        if len(bbox) == 4:
            if bbox[2] < bbox[0] or bbox[3] < bbox[1]:
                bbox_xyxy = bbox
            else:
                bbox_xyxy = [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
        else:
            bbox_xyxy = bbox
        
        score = pred['score']
        score_clipped = np.clip(score, 1e-7, 1 - 1e-7)
        logit = np.log(score_clipped / (1 - score_clipped))
        
        converted[img_id].append({
            'bbox': bbox_xyxy,
            'score': score_clipped,
            'logit': logit,
            'category_id': pred['category_id'],
            'uncertainty': pred.get('uncertainty', 0.0)
        })
    
    return converted

print("✅ Format conversion functions defined")

## 2. Load Model and Prepare Functions

In [ ]:
from groundingdino.util.inference import load_model, load_image, predict
from groundingdino.util import box_ops
import os

model_config = os.environ.get('GDINO_CONFIG', '/opt/program/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py')
model_weights = os.environ.get('GDINO_WEIGHTS', '/opt/program/GroundingDINO/weights/groundingdino_swint_ogc.pth')

model = load_model(model_config, model_weights)
model.to(CONFIG['device'])

TEXT_PROMPT = '. '.join(CONFIG['categories']) + '.'

print(f"Model loaded on {CONFIG['device']}")
print(f"Text prompt: {TEXT_PROMPT}")

dropout_modules = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Dropout) and ('class_embed' in name or 'bbox_embed' in name):
        dropout_modules.append(module)

print(f"Dropout modules in head: {len(dropout_modules)}")

In [ ]:
def normalize_label(label):
    synonyms = {'bike': 'bicycle', 'motorbike': 'motorcycle', 'pedestrian': 'person', 
                'stop sign': 'traffic sign', 'red light': 'traffic light'}
    label_lower = label.lower().strip()
    if label_lower in synonyms:
        return synonyms[label_lower]
    for cat in CONFIG['categories']:
        if cat in label_lower:
            return cat
    return label_lower

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def apply_nms(detections, iou_thresh=0.65):
    if len(detections) == 0:
        return []
    boxes_t = torch.tensor([d['bbox'] for d in detections], dtype=torch.float32)
    scores_t = torch.tensor([d['score'] for d in detections], dtype=torch.float32)
    keep = torchvision.ops.nms(boxes_t, scores_t, iou_thresh)
    return [detections[i] for i in keep.numpy()]

## 3. Inference Methods

In [ ]:
def inference_baseline(model, image_path, text_prompt, conf_thresh, device):
    """Method 1: Baseline single-pass without uncertainty"""
    model.eval()
    for module in dropout_modules:
        module.eval()
    
    image_source, image = load_image(str(image_path))
    boxes, scores, phrases = predict(model, image, text_prompt, conf_thresh, 0.25, device)
    
    if len(boxes) == 0:
        return []
    
    h, w = image_source.shape[:2]
    boxes_xyxy = box_ops.box_cxcywh_to_xyxy(boxes) * torch.tensor([w, h, w, h])
    
    detections = []
    for box, score, phrase in zip(boxes_xyxy.cpu().numpy(), scores.cpu().numpy(), phrases):
        cat = normalize_label(phrase)
        if cat in CONFIG['categories']:
            score_clipped = np.clip(float(score), 1e-7, 1 - 1e-7)
            logit = np.log(score_clipped / (1 - score_clipped))
            detections.append({
                'bbox': box.tolist(),
                'score': score_clipped,
                'logit': logit,
                'category': cat,
                'uncertainty': 0.0
            })
    
    return apply_nms(detections, CONFIG['nms_threshold'])

print("Method 1: Baseline defined")

In [ ]:
def inference_mc_dropout(model, image_path, text_prompt, conf_thresh, device, K=5):
    """Method: MC-Dropout with K forward passes"""
    model.eval()
    for module in dropout_modules:
        module.train()
    
    image_source, image = load_image(str(image_path))
    h, w = image_source.shape[:2]
    
    all_detections_k = []
    
    with torch.no_grad():
        for k in range(K):
            boxes, scores, phrases = predict(model, image, text_prompt, conf_thresh, 0.25, device)
            
            if len(boxes) == 0:
                all_detections_k.append([])
                continue
            
            boxes_xyxy = box_ops.box_cxcywh_to_xyxy(boxes) * torch.tensor([w, h, w, h])
            
            dets_k = []
            for box, score, phrase in zip(boxes_xyxy.cpu().numpy(), scores.cpu().numpy(), phrases):
                cat = normalize_label(phrase)
                if cat in CONFIG['categories']:
                    score_clipped = np.clip(float(score), 1e-7, 1 - 1e-7)
                    dets_k.append({
                        'bbox': box.tolist(),
                        'score': score_clipped,
                        'category': cat
                    })
            all_detections_k.append(dets_k)
    
    # Align detections across passes
    if len(all_detections_k) == 0 or all(len(d) == 0 for d in all_detections_k):
        return []
    
    # Use first pass as reference
    ref_dets = all_detections_k[0]
    
    aggregated = []
    for ref_det in ref_dets:
        scores_aligned = [ref_det['score']]
        
        for k in range(1, K):
            best_iou = 0
            best_score = None
            for det_k in all_detections_k[k]:
                if det_k['category'] != ref_det['category']:
                    continue
                iou = compute_iou(ref_det['bbox'], det_k['bbox'])
                if iou > best_iou:
                    best_iou = iou
                    best_score = det_k['score']
            
            if best_iou >= 0.5 and best_score is not None:
                scores_aligned.append(best_score)
        
        mean_score = np.mean(scores_aligned)
        variance = np.var(scores_aligned) if len(scores_aligned) > 1 else 0.0
        
        mean_score_clipped = np.clip(mean_score, 1e-7, 1 - 1e-7)
        logit = np.log(mean_score_clipped / (1 - mean_score_clipped))
        
        aggregated.append({
            'bbox': ref_det['bbox'],
            'score': mean_score_clipped,
            'logit': logit,
            'category': ref_det['category'],
            'uncertainty': variance
        })
    
    return apply_nms(aggregated, CONFIG['nms_threshold'])

print("Method: MC-Dropout defined")

In [ ]:
# Decoder variance inference function

def inference_decoder_variance(model, image_path, text_prompt, conf_thresh, device):
    """Method: Variance across decoder layers (single-pass)"""
    model.eval()
    for module in dropout_modules:
        module.eval()
    
    image_source, image = load_image(str(image_path))
    h, w = image_source.shape[:2]
    
    layer_logits = []
    
    def hook_fn(module, input, output):
        if isinstance(output, tuple) and len(output) > 0:
            layer_logits.append(output[0].detach() if hasattr(output[0], 'detach') else output[0])
        elif hasattr(output, 'detach'):
            layer_logits.append(output.detach())
    
    hooks = []
    for name, module in model.named_modules():
        if 'decoder.layers' in name and name.count('.') == 3 and name.split('.')[-1].isdigit():
            hooks.append(module.register_forward_hook(hook_fn))
    
    boxes, scores, phrases = predict(model, image, text_prompt, conf_thresh, 0.25, device)
    
    for hook in hooks:
        hook.remove()
    
    if len(boxes) == 0:
        return []
    
    boxes_xyxy = box_ops.box_cxcywh_to_xyxy(boxes) * torch.tensor([w, h, w, h])
    
    detections = []
    for idx, (box, score, phrase) in enumerate(zip(boxes_xyxy.cpu().numpy(), scores.cpu().numpy(), phrases)):
        cat = normalize_label(phrase)
        if cat in CONFIG['categories']:
            score_clipped = np.clip(float(score), 1e-7, 1 - 1e-7)
            logit = np.log(score_clipped / (1 - score_clipped))
            
            uncertainty = 0.0
            layer_uncertainties_list = []
            
            if len(layer_logits) > 0:
                layer_scores = []
                
                for layer_emb in layer_logits:
                    if idx < layer_emb.shape[0]:
                        query_emb = layer_emb[idx, 0, :]
                        emb_norm = torch.norm(query_emb).item()
                        layer_score = 1.0 / (1.0 + np.exp(-emb_norm / 10.0))
                        layer_scores.append(layer_score)
                
                if len(layer_scores) > 1:
                    uncertainty = np.var(layer_scores)
                    layer_uncertainties_list = layer_scores
            
            detections.append({
                'bbox': box.tolist(),
                'score': score_clipped,
                'logit': logit,
                'category': cat,
                'uncertainty': uncertainty,
                'layer_uncertainties': layer_uncertainties_list,
                'layer_count': len(layer_uncertainties_list)
            })
    
    return apply_nms(detections, CONFIG['nms_threshold'])

print("Method: Decoder variance defined (with layer_uncertainties)")

## 4. Inference on val_calib to Adjust Temperatures

### 4.1 Optimization Strategy

**If cached predictions are available**:
- Baseline → Loaded from Phase 2
- MC-Dropout → Loaded from Phase 3
- Only Decoder Variance is calculated (new method)

**If cached predictions are NOT available**:
- Full inference is run for all methods

In [ ]:
val_eval_json = DATA_DIR / 'bdd100k_coco/val_eval.json'
image_dir = DATA_DIR / 'bdd100k/bdd100k/bdd100k/images/100k/val'

coco_eval_full = COCO(str(val_eval_json))
img_ids_all = coco_eval_full.getImgIds()

# Split inteligente de val_eval (2000 images):
# - first 500 para calibración (ajustar temperaturas)
# - remaining 1500 para evaluación final
img_ids_calib = img_ids_all[:500]
img_ids_eval_final = img_ids_all[500:]

print(f"📊 ESTRATEGIA DE SPLITS:")
print(f"  Dataset: val_eval.json (2,000 images)")
print(f"  ├─ Calibration: {len(img_ids_calib)} images (first 500)")
print(f"  └─ Evaluation:  {len(img_ids_eval_final)} images (remaining 1,500)")

# Crear COCO object para calibración
coco_calib = COCO(str(val_eval_json))

print(f"\nProcesando {len(img_ids_calib)} images para ajustar temperaturas...")

methods_calib_data = {
    'baseline': [],
    'mc_dropout': [],
    'decoder_variance': []
}

# Contadores para diagnóstico
counters = {
    'baseline_cached': 0,
    'baseline_computed': 0,
    'mc_cached': 0,
    'mc_computed': 0
}

# ============================================================================
# OPTIMIZACIÓN: Usar predicciones cacheadas si están availables
# ============================================================================

# Convertir predicciones cacheadas a formato útil
baseline_by_img = {}
mc_by_img = {}

if cached_predictions['baseline']:
    print("\n✅ Using predictions Baseline cached from Phase 2")
    baseline_by_img = convert_baseline_predictions(cached_predictions['baseline'], {})
    print(f"   → {len(baseline_by_img)} images indexed")
    
if cached_predictions['mc_dropout']:
    print("✅ Using predictions MC-Dropout cached from Phase 3")
    mc_by_img = convert_mc_predictions(cached_predictions['mc_dropout'], {})
    print(f"   → {len(mc_by_img)} images indexed")

# Verificar overlap con first 500 images de val_eval
calib_500 = set(img_ids_calib)
baseline_overlap = set(baseline_by_img.keys()) & calib_500
mc_overlap = set(mc_by_img.keys()) & calib_500

print(f"\n🔍 OVERLAP CON CALIBRACIÓN (first 500 de val_eval):")
print(f"   Baseline cached: {len(baseline_overlap)}/500 images ({len(baseline_overlap)/500*100:.1f}%)")
print(f"   MC-Dropout cached: {len(mc_overlap)}/500 images ({len(mc_overlap)/500*100:.1f}%)")

if len(baseline_overlap) < 500:
    print(f"   ⚠️  {500 - len(baseline_overlap)} images de baseline se calcularán desde cero")
if len(mc_overlap) < 500:
    print(f"   ⚠️  {500 - len(mc_overlap)} images de MC-Dropout se calcularán desde cero")
    print(f"   ⏱️  Estimated time: ~{(500 - len(mc_overlap)) * 1.8 / 60:.1f} minutes")

### 🐛 DEBUG: Test inference_decoder_variance with 1 image

In [ ]:
# 🐛 DEBUG CELL - Run only ONE image to diagnose hooks

print("🐛 DEBUG: Testing inference_decoder_variance with one image...")

# Take the first calibration image
test_img_id = img_ids_calib[0]
test_img_info = coco_calib.loadImgs(test_img_id)[0]
test_img_path = image_dir / test_img_info['file_name']

print(f"   Test image: {test_img_info['file_name']} (ID: {test_img_id})")
print(f"   Path: {test_img_path}")
print(f"   Exists: {test_img_path.exists()}")

if test_img_path.exists():
    print(f"\n🔍 Running inference_decoder_variance...")
    test_preds = inference_decoder_variance(model, test_img_path, TEXT_PROMPT, CONFIG['conf_threshold'], CONFIG['device'])
    
    print(f"\n📊 RESULTS:")
    print(f"   Detections: {len(test_preds)}")
    
    if len(test_preds) > 0:
        print(f"\n   First detection:")
        first_pred = test_preds[0]
        for key, value in first_pred.items():
            print(f"      {key}: {value}")
        
        # Check layer_uncertainties
        if 'layer_uncertainties' in first_pred:
            layer_unc = first_pred['layer_uncertainties']
            print(f"\n   ✅ layer_uncertainties present: {len(layer_unc)} values")
            if len(layer_unc) > 0:
                print(f"      Values: {layer_unc}")
            else:
                print(f"      ⚠️  EMPTY - This is the issue to fix")
        else:
            print(f"\n   ❌ layer_uncertainties NOT in output")
    else:
        print(f"   ⚠️  No objects detected in this image")
else:
    print(f"   ❌ Image not found")

print(f"\n{'='*70}")
"completed"

In [ ]:
# Process calibration images
for img_id in tqdm(img_ids_calib, desc="Processing calibration"):
    img_info = coco_calib.loadImgs(img_id)[0]
    img_path = image_dir / img_info['file_name']
    
    if not img_path.exists():
        continue
    
    gt_anns = coco_calib.loadAnns(coco_calib.getAnnIds(imgIds=img_id))
    
    # Baseline Method
    if img_id in baseline_by_img:
        preds_baseline = baseline_by_img[img_id]
        counters['baseline_cached'] += 1
    else:
        counters['baseline_computed'] += 1
        preds_baseline_raw = inference_baseline(model, img_path, TEXT_PROMPT, CONFIG['conf_threshold'], CONFIG['device'])
        preds_baseline = []
        for pred in preds_baseline_raw:
            cat_id = CONFIG['categories'].index(pred['category']) + 1
            preds_baseline.append({
                'bbox': pred['bbox'],
                'score': pred['score'],
                'logit': pred['logit'],
                'category_id': cat_id,
                'uncertainty': pred['uncertainty']
            })
    
    # Label as TP/FP
    for pred in preds_baseline:
        is_tp = 0
        cat_id = pred['category_id']
        cat = CONFIG['categories'][cat_id - 1] if 1 <= cat_id <= len(CONFIG['categories']) else ''
        
        for gt in gt_anns:
            if gt['category_id'] != cat_id:
                continue
            gt_box = gt['bbox']
            gt_box_xyxy = [gt_box[0], gt_box[1], gt_box[0] + gt_box[2], gt_box[1] + gt_box[3]]
            if compute_iou(pred['bbox'], gt_box_xyxy) >= CONFIG['iou_matching']:
                is_tp = 1
                break
        
        methods_calib_data['baseline'].append({
            'logit': pred['logit'],
            'score': pred['score'],
            'category': cat,
            'uncertainty': pred['uncertainty'],
            'is_tp': is_tp
        })
    
    # MC-Dropout Method
    if img_id in mc_by_img:
        preds_mc = mc_by_img[img_id]
        counters['mc_cached'] += 1
    else:
        counters['mc_computed'] += 1
        preds_mc_raw = inference_mc_dropout(model, img_path, TEXT_PROMPT, CONFIG['conf_threshold'], CONFIG['device'], CONFIG['K_mc'])
        preds_mc = []
        for pred in preds_mc_raw:
            cat_id = CONFIG['categories'].index(pred['category']) + 1
            preds_mc.append({
                'bbox': pred['bbox'],
                'score': pred['score'],
                'logit': pred['logit'],
                'category_id': cat_id,
                'uncertainty': pred['uncertainty']
            })
    
    for pred in preds_mc:
        is_tp = 0
        cat_id = pred['category_id']
        cat = CONFIG['categories'][cat_id - 1] if 1 <= cat_id <= len(CONFIG['categories']) else ''
        
        for gt in gt_anns:
            if gt['category_id'] != cat_id:
                continue
            gt_box = gt['bbox']
            gt_box_xyxy = [gt_box[0], gt_box[1], gt_box[0] + gt_box[2], gt_box[1] + gt_box[3]]
            if compute_iou(pred['bbox'], gt_box_xyxy) >= CONFIG['iou_matching']:
                is_tp = 1
                break
        
        methods_calib_data['mc_dropout'].append({
            'logit': pred['logit'],
            'score': pred['score'],
            'category': cat,
            'uncertainty': pred['uncertainty'],
            'is_tp': is_tp
        })
    
    # Decoder Variance Method (always calculated, it's new)
    preds_dec = inference_decoder_variance(model, img_path, TEXT_PROMPT, CONFIG['conf_threshold'], CONFIG['device'])
    for pred in preds_dec:
        is_tp = 0
        cat_id = CONFIG['categories'].index(pred['category']) + 1
        cat = pred['category']
        
        for gt in gt_anns:
            if gt['category_id'] != cat_id:
                continue
            gt_box = gt['bbox']
            gt_box_xyxy = [gt_box[0], gt_box[1], gt_box[0] + gt_box[2], gt_box[1] + gt_box[3]]
            if compute_iou(pred['bbox'], gt_box_xyxy) >= CONFIG['iou_matching']:
                is_tp = 1
                break
        
        methods_calib_data['decoder_variance'].append({
            'logit': pred['logit'],
            'score': pred['score'],
            'category': cat,
            'uncertainty': pred['uncertainty'],
            'is_tp': is_tp
        })

print(f"\n📊 PROCESSING STATISTICS:")
print(f"   Baseline: {counters['baseline_cached']} cached, {counters['baseline_computed']} computed")
print(f"   MC-Dropout: {counters['mc_cached']} cached, {counters['mc_computed']} computed")

# Save calibration data
for method_name, data in methods_calib_data.items():
    df = pd.DataFrame(data)
    df.to_csv(OUTPUT_DIR / f'calib_{method_name}.csv', index=False)
    print(f"\n{method_name}: {len(df)} detections, TP={df['is_tp'].sum()}")

print("\n✅ Calibration data saved")

# Final diagnosis
print(f"\n🔍 FINAL DIAGNOSIS - Verifying calibration CSVs:")
df_baseline = pd.read_csv(OUTPUT_DIR / 'calib_baseline.csv')
df_mc = pd.read_csv(OUTPUT_DIR / 'calib_mc_dropout.csv')
print(f"   Baseline: {len(df_baseline)} records, mean uncertainty={df_baseline['uncertainty'].mean():.6f}")
print(f"   MC-Dropout: {len(df_mc)} records, mean uncertainty={df_mc['uncertainty'].mean():.6f}")

print(f"\n   First 5 logits from each method:")
print(f"   Baseline:   {df_baseline['logit'].head().tolist()}")
print(f"   MC-Dropout: {df_mc['logit'].head().tolist()}")

if df_baseline['logit'].head(10).equals(df_mc['logit'].head(10)):
    print(f"   ⚠️  First 10 logits are identical (may be coincidence or issue)")
else:
    print(f"   ✅ Logits are different")
    
if df_baseline['uncertainty'].equals(df_mc['uncertainty']):
    print(f"   ⚠️  Uncertainties are identical")
else:
    print(f"   ✅ Uncertainties are different")

## 5. Optimizar Temperaturas

In [ ]:
from scipy.optimize import minimize

def nll_loss(T, logits, labels):
    T = max(T, 0.01)
    probs = sigmoid(logits / T)
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    return -np.mean(labels * np.log(probs) + (1 - labels) * np.log(1 - probs))

if cached_predictions['temperatures'] and 'optimal_temperature' in cached_predictions['temperatures']:
    print("✅ Using optimized temperature from Phase 4")
    T_baseline = cached_predictions['temperatures']['optimal_temperature']
    
    df_baseline = pd.read_csv(OUTPUT_DIR / 'calib_baseline.csv')
    logits_baseline = df_baseline['logit'].values
    labels_baseline = df_baseline['is_tp'].values
    
    nll_before = nll_loss(1.0, logits_baseline, labels_baseline)
    nll_after = nll_loss(T_baseline, logits_baseline, labels_baseline)
    
    temperatures = {
        'baseline': {
            'T': T_baseline,
            'nll_before': nll_before,
            'nll_after': nll_after,
            'source': 'cached_from_phase4'
        }
    }
    
    print(f"  baseline: T={T_baseline:.4f}, NLL: {nll_before:.4f} → {nll_after:.4f} (cached)")
    
else:
    print("⚙️  Calculating temperatures from scratch...")
    temperatures = {}

# Calculate temperatures for MC-Dropout and Decoder Variance (always, they're new)
for method_name in ['mc_dropout', 'decoder_variance']:
    df = pd.read_csv(OUTPUT_DIR / f'calib_{method_name}.csv')
    logits = df['logit'].values
    labels = df['is_tp'].values
    
    nll_before = nll_loss(1.0, logits, labels)
    result = minimize(lambda T: nll_loss(T, logits, labels), x0=1.0, bounds=[(0.01, 10.0)], method='L-BFGS-B')
    T_opt = result.x[0]
    nll_after = result.fun
    
    temperatures[method_name] = {
        'T': T_opt,
        'nll_before': nll_before,
        'nll_after': nll_after,
        'source': 'calculated'
    }
    
    print(f"  {method_name}: T={T_opt:.4f}, NLL: {nll_before:.4f} → {nll_after:.4f}")

# If baseline temperature wasn't cached, calculate it
if 'baseline' not in temperatures:
    print("⚙️  Calculating temperature for baseline...")
    df = pd.read_csv(OUTPUT_DIR / 'calib_baseline.csv')
    logits = df['logit'].values
    labels = df['is_tp'].values
    
    nll_before = nll_loss(1.0, logits, labels)
    result = minimize(lambda T: nll_loss(T, logits, labels), x0=1.0, bounds=[(0.01, 10.0)], method='L-BFGS-B')
    T_opt = result.x[0]
    nll_after = result.fun
    
    temperatures['baseline'] = {
        'T': T_opt,
        'nll_before': nll_before,
        'nll_after': nll_after,
        'source': 'calculated'
    }
    
    print(f"  baseline: T={T_opt:.4f}, NLL: {nll_before:.4f} → {nll_after:.4f}")

with open(OUTPUT_DIR / 'temperatures.json', 'w') as f:
    json.dump(temperatures, f, indent=2)

print(f"\n✅ Temperatures saved to: {OUTPUT_DIR / 'temperatures.json'}")

## 6. Evaluation on val_eval with COCO API

In [ ]:
# Run inference on val_eval (1500 images)
# This cell runs decoder_variance on ALL evaluation images

# ============================================================================
# Evaluation: Use remaining 1500 images from val_eval
# ============================================================================
# The first 500 were used for calibration, now we use the rest

print(f"\n{'='*70}")
print(f"EVALUATION ON VAL_EVAL (1,500 remaining images)")
print(f"{'='*70}")

coco_eval = COCO(str(val_eval_json))

print(f"Processing {len(img_ids_eval_final)} images for final evaluation...")

methods_results = {
    'baseline': [],
    'baseline_ts': [],
    'mc_dropout': [],
    'mc_dropout_ts': [],
    'decoder_variance': [],
    'decoder_variance_ts': []
}

# Load temperatures
with open(OUTPUT_DIR / 'temperatures.json', 'r') as f:
    temps = json.load(f)

print(f"\n📊 Temperatures to apply:")
for method, temp_info in temps.items():
    print(f"   {method}: T={temp_info['T']:.4f}")

# ============================================================================
# OPTIMIZATION: Build indices of cached predictions for evaluation
# ============================================================================

baseline_eval_by_img = {}
mc_eval_by_img = {}

if cached_predictions['baseline']:
    print("\n✅ Indexing cached Baseline predictions for evaluation")
    for pred in cached_predictions['baseline']:
        img_id = pred.get('image_id')
        if img_id in img_ids_eval_final:  # Only evaluation images (not calibration)
            if img_id not in baseline_eval_by_img:
                baseline_eval_by_img[img_id] = []
            baseline_eval_by_img[img_id].append(pred)
    print(f"   → {len(baseline_eval_by_img)} images with cached predictions")

if cached_predictions['mc_dropout']:
    print("✅ Indexing cached MC-Dropout predictions for evaluation")
    for pred in cached_predictions['mc_dropout']:
        img_id = pred.get('image_id')
        if img_id in img_ids_eval_final:  # Only evaluation images (not calibration)
            if img_id not in mc_eval_by_img:
                mc_eval_by_img[img_id] = []
            mc_eval_by_img[img_id].append(pred)
    print(f"   → {len(mc_eval_by_img)} images with cached predictions")

# Counters
eval_counters = {
    'baseline_cached': 0,
    'baseline_computed': 0,
    'mc_cached': 0,
    'mc_computed': 0
}

# Process evaluation images
for img_id in tqdm(img_ids_eval_final, desc="Processing evaluation"):
    img_info = coco_eval.loadImgs(img_id)[0]
    img_path = image_dir / img_info['file_name']
    
    if not img_path.exists():
        continue
    
    gt_anns = coco_eval.loadAnns(coco_eval.getAnnIds(imgIds=img_id))
    
    # ========================================================================
    # Baseline (without TS and with TS)
    # ========================================================================
    if img_id in baseline_eval_by_img:
        # Use cached
        eval_counters['baseline_cached'] += 1
        for pred in baseline_eval_by_img[img_id]:
            is_tp = 0
            bbox = pred['bbox']
            bbox_xyxy = [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
            
            for gt in gt_anns:
                if gt['category_id'] != pred['category_id']:
                    continue
                gt_box = gt['bbox']
                gt_box_xyxy = [gt_box[0], gt_box[1], gt_box[0] + gt_box[2], gt_box[1] + gt_box[3]]
                if compute_iou(bbox_xyxy, gt_box_xyxy) >= CONFIG['iou_matching']:
                    is_tp = 1
                    break
            
            score = pred['score']
            score_clipped = np.clip(score, 1e-7, 1 - 1e-7)
            logit = np.log(score_clipped / (1 - score_clipped))
            
            methods_results['baseline'].append({
                'image_id': img_id,
                'category_id': pred['category_id'],
                'bbox': pred['bbox'],
                'score': score_clipped,
                'logit': logit,
                'uncertainty': pred.get('uncertainty', 0.0),
                'is_tp': is_tp
            })
            
            # With TS
            score_ts = sigmoid(logit / temps['baseline']['T'])
            methods_results['baseline_ts'].append({
                'image_id': img_id,
                'category_id': pred['category_id'],
                'bbox': pred['bbox'],
                'score': score_ts,
                'logit': logit,
                'uncertainty': pred.get('uncertainty', 0.0),
                'is_tp': is_tp
            })
    else:
        # Compute from scratch
        eval_counters['baseline_computed'] += 1
        preds_baseline = inference_baseline(model, img_path, TEXT_PROMPT, CONFIG['conf_threshold'], CONFIG['device'])
        for pred in preds_baseline:
            cat_id = CONFIG['categories'].index(pred['category']) + 1
            is_tp = 0
            for gt in gt_anns:
                if gt['category_id'] != cat_id:
                    continue
                gt_box = gt['bbox']
                gt_box_xyxy = [gt_box[0], gt_box[1], gt_box[0] + gt_box[2], gt_box[1] + gt_box[3]]
                if compute_iou(pred['bbox'], gt_box_xyxy) >= CONFIG['iou_matching']:
                    is_tp = 1
                    break
            
            methods_results['baseline'].append({
                'image_id': img_id,
                'category_id': cat_id,
                'bbox': [pred['bbox'][0], pred['bbox'][1], pred['bbox'][2] - pred['bbox'][0], pred['bbox'][3] - pred['bbox'][1]],
                'score': pred['score'],
                'logit': pred['logit'],
                'uncertainty': pred['uncertainty'],
                'is_tp': is_tp
            })
            
            score_ts = sigmoid(pred['logit'] / temps['baseline']['T'])
            methods_results['baseline_ts'].append({
                'image_id': img_id,
                'category_id': cat_id,
                'bbox': [pred['bbox'][0], pred['bbox'][1], pred['bbox'][2] - pred['bbox'][0], pred['bbox'][3] - pred['bbox'][1]],
                'score': score_ts,
                'logit': pred['logit'],
                'uncertainty': pred['uncertainty'],
                'is_tp': is_tp
            })
    
    # ========================================================================
    # MC-Dropout (without TS and with TS)
    # ========================================================================
    if img_id in mc_eval_by_img:
        # Use cached
        eval_counters['mc_cached'] += 1
        for pred in mc_eval_by_img[img_id]:
            is_tp = 0
            bbox = pred['bbox']
            bbox_xyxy = [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
            
            for gt in gt_anns:
                if gt['category_id'] != pred['category_id']:
                    continue
                gt_box = gt['bbox']
                gt_box_xyxy = [gt_box[0], gt_box[1], gt_box[0] + gt_box[2], gt_box[1] + gt_box[3]]
                if compute_iou(bbox_xyxy, gt_box_xyxy) >= CONFIG['iou_matching']:
                    is_tp = 1
                    break
            
            score = pred['score']
            score_clipped = np.clip(score, 1e-7, 1 - 1e-7)
            logit = np.log(score_clipped / (1 - score_clipped))
            
            methods_results['mc_dropout'].append({
                'image_id': img_id,
                'category_id': pred['category_id'],
                'bbox': pred['bbox'],
                'score': score_clipped,
                'logit': logit,
                'uncertainty': pred.get('uncertainty', 0.0),
                'is_tp': is_tp
            })
            
            # With TS
            score_ts = sigmoid(logit / temps['mc_dropout']['T'])
            methods_results['mc_dropout_ts'].append({
                'image_id': img_id,
                'category_id': pred['category_id'],
                'bbox': pred['bbox'],
                'score': score_ts,
                'logit': logit,
                'uncertainty': pred.get('uncertainty', 0.0),
                'is_tp': is_tp
            })
    else:
        # Compute from scratch
        eval_counters['mc_computed'] += 1
        preds_mc = inference_mc_dropout(model, img_path, TEXT_PROMPT, CONFIG['conf_threshold'], CONFIG['device'], CONFIG['K_mc'])
        for pred in preds_mc:
            cat_id = CONFIG['categories'].index(pred['category']) + 1
            is_tp = 0
            for gt in gt_anns:
                if gt['category_id'] != cat_id:
                    continue
                gt_box = gt['bbox']
                gt_box_xyxy = [gt_box[0], gt_box[1], gt_box[0] + gt_box[2], gt_box[1] + gt_box[3]]
                if compute_iou(pred['bbox'], gt_box_xyxy) >= CONFIG['iou_matching']:
                    is_tp = 1
                    break
            
            methods_results['mc_dropout'].append({
                'image_id': img_id,
                'category_id': cat_id,
                'bbox': [pred['bbox'][0], pred['bbox'][1], pred['bbox'][2] - pred['bbox'][0], pred['bbox'][3] - pred['bbox'][1]],
                'score': pred['score'],
                'logit': pred['logit'],
                'uncertainty': pred['uncertainty'],
                'is_tp': is_tp
            })
            
            score_ts = sigmoid(pred['logit'] / temps['mc_dropout']['T'])
            methods_results['mc_dropout_ts'].append({
                'image_id': img_id,
                'category_id': cat_id,
                'bbox': [pred['bbox'][0], pred['bbox'][1], pred['bbox'][2] - pred['bbox'][0], pred['bbox'][3] - pred['bbox'][1]],
                'score': score_ts,
                'logit': pred['logit'],
                'uncertainty': pred['uncertainty'],
                'is_tp': is_tp
            })
    
    # ========================================================================
    # Decoder variance (always calculated, it's new)
    # ========================================================================
    preds_dec = inference_decoder_variance(model, img_path, TEXT_PROMPT, CONFIG['conf_threshold'], CONFIG['device'])
    for pred in preds_dec:
        cat_id = CONFIG['categories'].index(pred['category']) + 1
        is_tp = 0
        for gt in gt_anns:
            if gt['category_id'] != cat_id:
                continue
            gt_box = gt['bbox']
            gt_box_xyxy = [gt_box[0], gt_box[1], gt_box[0] + gt_box[2], gt_box[1] + gt_box[3]]
            if compute_iou(pred['bbox'], gt_box_xyxy) >= CONFIG['iou_matching']:
                is_tp = 1
                break
        
        methods_results['decoder_variance'].append({
            'image_id': img_id,
            'category_id': cat_id,
            'bbox': [pred['bbox'][0], pred['bbox'][1], pred['bbox'][2] - pred['bbox'][0], pred['bbox'][3] - pred['bbox'][1]],
            'score': pred['score'],
            'logit': pred['logit'],
            'uncertainty': pred['uncertainty'],
            'layer_uncertainties': pred.get('layer_uncertainties', []),
            'is_tp': is_tp
        })
        
        score_ts = sigmoid(pred['logit'] / temps['decoder_variance']['T'])
        methods_results['decoder_variance_ts'].append({
            'image_id': img_id,
            'category_id': cat_id,
            'bbox': [pred['bbox'][0], pred['bbox'][1], pred['bbox'][2] - pred['bbox'][0], pred['bbox'][3] - pred['bbox'][1]],
            'score': score_ts,
            'logit': pred['logit'],
            'uncertainty': pred['uncertainty'],
            'layer_uncertainties': pred.get('layer_uncertainties', []),
            'is_tp': is_tp
        })

# Final statistics
print(f"\n📊 EVALUATION STATISTICS:")
print(f"   Baseline: {eval_counters['baseline_cached']} cached, {eval_counters['baseline_computed']} computed")
print(f"   MC-Dropout: {eval_counters['mc_cached']} cached, {eval_counters['mc_computed']} computed")

# Save results (CSV and JSON)
for method_name, results in methods_results.items():
    # Save CSV
    df = pd.DataFrame(results)
    df.to_csv(OUTPUT_DIR / f'eval_{method_name}.csv', index=False)
    
    # Save JSON (format compatible with RQ1)
    with open(OUTPUT_DIR / f'eval_{method_name}.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n{method_name}: {len(df)} detections")

print(f"\n✅ Evaluation results saved (CSV + JSON)")

## 7. Calculate Detection Metrics (mAP)

In [ ]:
detection_metrics = {}

for method_name in methods_results.keys():
    print(f"\nEvaluating {method_name}...")
    
    # Load predictions in COCO format
    preds_file = OUTPUT_DIR / f'eval_{method_name}.json'
    
    if os.path.getsize(preds_file) > 0:
        coco_dt = coco_eval.loadRes(str(preds_file))
        coco_eval_obj = COCOeval(coco_eval, coco_dt, 'bbox')
        coco_eval_obj.evaluate()
        coco_eval_obj.accumulate()
        coco_eval_obj.summarize()
        
        detection_metrics[method_name] = {
            'mAP': coco_eval_obj.stats[0],
            'AP50': coco_eval_obj.stats[1],
            'AP75': coco_eval_obj.stats[2],
            'AP_small': coco_eval_obj.stats[3],
            'AP_medium': coco_eval_obj.stats[4],
            'AP_large': coco_eval_obj.stats[5]
        }
        
        # mAP by class
        per_class_ap = {}
        for cat_id, cat_name in enumerate(CONFIG['categories'], 1):
            coco_eval_obj.params.catIds = [cat_id]
            coco_eval_obj.evaluate()
            coco_eval_obj.accumulate()
            per_class_ap[cat_name] = coco_eval_obj.stats[0]
        
        detection_metrics[method_name]['per_class'] = per_class_ap
    else:
        detection_metrics[method_name] = {'mAP': 0.0, 'AP50': 0.0, 'AP75': 0.0}

with open(OUTPUT_DIR / 'detection_metrics.json', 'w') as f:
    json.dump(detection_metrics, f, indent=2)

print("\nDetection metrics saved")

## 8. Detection Performance Comparison Table

In [ ]:
with open(OUTPUT_DIR / 'detection_metrics.json', 'r') as f:
    det_metrics = json.load(f)

# Create comparison table
rows = []
for method_name, metrics in det_metrics.items():
    row = {
        'Method': method_name,
        'mAP': metrics.get('mAP', 0.0),
        'AP50': metrics.get('AP50', 0.0),
        'AP75': metrics.get('AP75', 0.0)
    }
    
    # Add mAP by main class
    if 'per_class' in metrics:
        for cat in ['person', 'car', 'truck', 'traffic_light', 'traffic_sign']:
            cat_key = cat.replace('_', ' ')
            row[f'AP_{cat}'] = metrics['per_class'].get(cat_key, 0.0)
    
    rows.append(row)

df_detection = pd.DataFrame(rows)
df_detection.to_csv(OUTPUT_DIR / 'detection_comparison.csv', index=False)

print("\n" + "="*80)
print("DETECTION PERFORMANCE COMPARISON")
print("="*80)
print(df_detection.to_string(index=False))
print("="*80)

## 9. Calculate Calibration Metrics

In [ ]:
def compute_calibration_metrics(logits, labels, T=1.0, n_bins=10):
    probs = sigmoid(logits / T)
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    
    nll = -np.mean(labels * np.log(probs) + (1 - labels) * np.log(1 - probs))
    brier = np.mean((probs - labels) ** 2)
    
    bins = np.linspace(0, 1, n_bins + 1)
    digitized = np.digitize(probs, bins) - 1
    
    ece = 0.0
    bin_data = []
    
    for i in range(n_bins):
        mask = digitized == i
        if mask.sum() > 0:
            conf = probs[mask].mean()
            acc = labels[mask].mean()
            gap = abs(conf - acc)
            ece += gap * mask.sum() / len(probs)
            bin_data.append({
                'bin': i,
                'confidence': conf,
                'accuracy': acc,
                'count': mask.sum()
            })
    
    return {'NLL': nll, 'Brier': brier, 'ECE': ece, 'bin_data': bin_data}

calibration_metrics = {}

for method_name in methods_results.keys():
    df = pd.read_csv(OUTPUT_DIR / f'eval_{method_name}.csv')
    logits = df['logit'].values
    labels = df['is_tp'].values
    
    if '_ts' not in method_name:
        metrics = compute_calibration_metrics(logits, labels, T=1.0, n_bins=CONFIG['n_bins'])
        calibration_metrics[method_name] = metrics
    else:
        base_method = method_name.replace('_ts', '')
        T = temps[base_method]['T']
        metrics = compute_calibration_metrics(logits, labels, T=T, n_bins=CONFIG['n_bins'])
        calibration_metrics[method_name] = metrics
    
    print(f"{method_name}: NLL={metrics['NLL']:.4f}, Brier={metrics['Brier']:.4f}, ECE={metrics['ECE']:.4f}")

with open(OUTPUT_DIR / 'calibration_metrics.json', 'w') as f:
    cal_save = {}
    for k, v in calibration_metrics.items():
        cal_save[k] = {
            'NLL': v['NLL'],
            'Brier': v['Brier'],
            'ECE': v['ECE']
        }
    json.dump(cal_save, f, indent=2)

print("\nCalibration metrics saved")

## 10. Calibration Performance Comparison Table

In [ ]:
rows_calib = []
for method_name, metrics in calibration_metrics.items():
    rows_calib.append({
        'Method': method_name,
        'NLL': metrics['NLL'],
        'Brier': metrics['Brier'],
        'ECE': metrics['ECE']
    })

df_calibration = pd.DataFrame(rows_calib)
df_calibration.to_csv(OUTPUT_DIR / 'calibration_comparison.csv', index=False)

print("\n" + "="*80)
print("CALIBRATION PERFORMANCE COMPARISON")
print("="*80)
print(df_calibration.to_string(index=False))
print("="*80)
print("\nInterpretation:")
print("  ↓ Lower is better for NLL, Brier, ECE")
print("  If method+TS < method: TS improved calibration")

## 11. Reliability Diagrams

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

method_pairs = [
    ('baseline', 'baseline_ts'),
    ('mc_dropout', 'mc_dropout_ts'),
    ('decoder_variance', 'decoder_variance_ts')
]

for idx, (method_before, method_after) in enumerate(method_pairs):
    ax = axes[idx * 2]
    
    bin_data = calibration_metrics[method_before]['bin_data']
    if len(bin_data) > 0:
        confidences = [b['confidence'] for b in bin_data]
        accuracies = [b['accuracy'] for b in bin_data]
        counts = [b['count'] for b in bin_data]
        
        ax.bar(range(len(confidences)), accuracies, alpha=0.3, label='Accuracy', color='blue')
        ax.plot(range(len(confidences)), confidences, 'o-', label='Confidence', color='red', markersize=8)
        ax.plot([0, len(confidences)-1], [0, 1], 'k--', alpha=0.3, label='Perfect calibration')
        ax.set_xlabel('Confidence bin')
        ax.set_ylabel('Proportion')
        ax.set_title(f'{method_before}\nECE={calibration_metrics[method_before]["ECE"]:.4f}')
        ax.legend()
        ax.grid(alpha=0.3)
    
    ax = axes[idx * 2 + 1]
    bin_data = calibration_metrics[method_after]['bin_data']
    if len(bin_data) > 0:
        confidences = [b['confidence'] for b in bin_data]
        accuracies = [b['accuracy'] for b in bin_data]
        
        ax.bar(range(len(confidences)), accuracies, alpha=0.3, label='Accuracy', color='blue')
        ax.plot(range(len(confidences)), confidences, 'o-', label='Confidence', color='red', markersize=8)
        ax.plot([0, len(confidences)-1], [0, 1], 'k--', alpha=0.3, label='Perfect calibration')
        ax.set_xlabel('Confidence bin')
        ax.set_ylabel('Proportion')
        ax.set_title(f'{method_after}\nECE={calibration_metrics[method_after]["ECE"]:.4f}')
        ax.legend()
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reliability_diagrams.png', dpi=150, bbox_inches='tight')
print(f"Reliability diagrams saved to: {OUTPUT_DIR / 'reliability_diagrams.png'}")
plt.close()

## 12. Risk-Coverage Analysis

## 13. Uncertainty Metrics: AUROC TP vs FP

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

uncertainty_auroc = {}

# Solo métodos with uncertainty (MC-Dropout y Decoder Variance)
uncertainty_methods = ['mc_dropout', 'mc_dropout_ts', 'decoder_variance', 'decoder_variance_ts']

print("="*80)
print("AUROC: Does uncertainty detect errors (FP)?")
print("="*80)
print("\nObjective: Use uncertainty to distinguish FP (errors) from TP (correct)")
print("Interpretation: AUROC > 0.5 (random), ideal ≥ 0.7")
print("-"*80)

for method_name in uncertainty_methods:
    df = pd.read_csv(OUTPUT_DIR / f'eval_{method_name}.csv')
    
    if len(df) > 0 and 'uncertainty' in df.columns:
        uncertainties = df['uncertainty'].values
        is_tp = df['is_tp'].values
        
        # Verificar que hay TPs y FPs
        if len(np.unique(is_tp)) > 1 and len(uncertainties) > 0:
            # AUROC: predecir FP (error) usando incertidumbre
            # Invertir labels: 1=FP (error), 0=TP (correcto)
            is_fp = 1 - is_tp
            
            try:
                auroc = roc_auc_score(is_fp, uncertainties)
                
                # Estadísticas de incertidumbre
                unc_tp = uncertainties[is_tp == 1]
                unc_fp = uncertainties[is_tp == 0]
                
                mean_unc_tp = unc_tp.mean() if len(unc_tp) > 0 else 0.0
                mean_unc_fp = unc_fp.mean() if len(unc_fp) > 0 else 0.0
                
                uncertainty_auroc[method_name] = {
                    'auroc': auroc,
                    'mean_unc_tp': mean_unc_tp,
                    'mean_unc_fp': mean_unc_fp,
                    'n_tp': int(is_tp.sum()),
                    'n_fp': int((1 - is_tp).sum())
                }
                
                print(f"\n{method_name}:")
                print(f"  AUROC (FP detection): {auroc:.4f}")
                print(f"  Mean uncertainty TP:  {mean_unc_tp:.6f}")
                print(f"  Mean uncertainty FP:  {mean_unc_fp:.6f}")
                print(f"  Ratio (FP/TP):        {mean_unc_fp/mean_unc_tp if mean_unc_tp > 0 else 0:.2f}x")
                print(f"  Samples: {int(is_tp.sum())} TP, {int((1-is_tp).sum())} FP")
                
            except Exception as e:
                print(f"\n{method_name}: Error calculating AUROC - {e}")
        else:
            print(f"\n{method_name}: Insufficient data for AUROC")

# Guardar resultados
with open(OUTPUT_DIR / 'uncertainty_auroc.json', 'w') as f:
    json.dump(uncertainty_auroc, f, indent=2)

print("\n" + "="*80)
print(f"Results saved to: {OUTPUT_DIR / 'uncertainty_auroc.json'}")


In [ ]:
# Comparative table AUROC
rows_auroc = []
for method_name, metrics in uncertainty_auroc.items():
    rows_auroc.append({
        'Method': method_name,
        'AUROC (FP detection) ↑': metrics['auroc'],
        'Mean Unc. TP': metrics['mean_unc_tp'],
        'Mean Unc. FP': metrics['mean_unc_fp'],
        'Ratio (FP/TP)': metrics['mean_unc_fp'] / metrics['mean_unc_tp'] if metrics['mean_unc_tp'] > 0 else 0
    })

df_auroc = pd.DataFrame(rows_auroc)
df_auroc.to_csv(OUTPUT_DIR / 'uncertainty_auroc_comparison.csv', index=False)

print("\n" + "="*80)
print("COMPARISON TABLE: AUROC TP vs FP")
print("="*80)
print(df_auroc.to_string(index=False))
print("="*80)
print("\nInterpretation:")
print("  ↑ Higher AUROC = better error detection")
print("  Ratio (FP/TP) > 1 = higher uncertainty in errors (desirable)")
print("  AUROC ≥ 0.7 = uncertainty useful for selective rejection")


In [ ]:
# Visualization: Uncertainty Distributions and ROC curves
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

methods_to_plot = ['mc_dropout', 'decoder_variance']
colors_methods = {'mc_dropout': 'blue', 'decoder_variance': 'green'}

for idx, method_name in enumerate(methods_to_plot):
    # Row 1: Uncertainty Distributions (TP vs FP)
    ax_dist = axes[0, idx]
    
    df = pd.read_csv(OUTPUT_DIR / f'eval_{method_name}.csv')
    if len(df) > 0 and 'uncertainty' in df.columns:
        unc_tp = df[df['is_tp'] == 1]['uncertainty'].values
        unc_fp = df[df['is_tp'] == 0]['uncertainty'].values
        
        ax_dist.hist(unc_tp, bins=50, alpha=0.6, label=f'TP (n={len(unc_tp)})', color='green', density=True)
        ax_dist.hist(unc_fp, bins=50, alpha=0.6, label=f'FP (n={len(unc_fp)})', color='red', density=True)
        ax_dist.axvline(unc_tp.mean(), color='green', linestyle='--', linewidth=2, label=f'Mean TP: {unc_tp.mean():.4f}')
        ax_dist.axvline(unc_fp.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean FP: {unc_fp.mean():.4f}')
        ax_dist.set_xlabel('Uncertainty', fontsize=11)
        ax_dist.set_ylabel('Density', fontsize=11)
        ax_dist.set_title(f'{method_name.replace("_", " ").title()}\nUncertainty Distribution', fontsize=12, fontweight='bold')
        ax_dist.legend(fontsize=9)
        ax_dist.grid(alpha=0.3)
    
    # Row 2: ROC curves
    ax_roc = axes[1, idx]
    
    if method_name in uncertainty_auroc:
        is_tp = df['is_tp'].values
        is_fp = 1 - is_tp
        uncertainties = df['uncertainty'].values
        
        fpr, tpr, thresholds = roc_curve(is_fp, uncertainties)
        auroc = uncertainty_auroc[method_name]['auroc']
        
        ax_roc.plot(fpr, tpr, linewidth=2, label=f'AUROC = {auroc:.4f}', color=colors_methods[method_name])
        ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (0.5)')
        ax_roc.set_xlabel('False Positive Rate', fontsize=11)
        ax_roc.set_ylabel('True Positive Rate', fontsize=11)
        ax_roc.set_title(f'{method_name.replace("_", " ").title()}\nROC Curve (FP Detection)', fontsize=12, fontweight='bold')
        ax_roc.legend(fontsize=10)
        ax_roc.grid(alpha=0.3)
        ax_roc.set_xlim([0, 1])
        ax_roc.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'uncertainty_analysis.png', dpi=150, bbox_inches='tight')
print(f"\nUncertainty visualization saved to: {OUTPUT_DIR / 'uncertainty_analysis.png'}")
plt.close()


## 12. Risk-Coverage Analysis

In [ ]:
def compute_risk_coverage(df, uncertainty_col='uncertainty'):
    """Calculate risk-coverage curve"""
    df_sorted = df.sort_values(uncertainty_col, ascending=False).reset_index(drop=True)
    
    coverages = []
    risks = []
    
    for i in range(1, len(df_sorted) + 1):
        coverage = i / len(df_sorted)
        risk = 1 - df_sorted.iloc[:i]['is_tp'].mean()
        coverages.append(coverage)
        risks.append(risk)
    
    # AUC (area under the curve)
    auc = np.trapz(risks, coverages)
    
    return coverages, risks, auc

# Calculate risk-coverage for methods with uncertainty
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

methods_with_uncertainty = ['mc_dropout', 'mc_dropout_ts', 'decoder_variance', 'decoder_variance_ts']
colors = ['blue', 'cyan', 'red', 'orange']

risk_coverage_results = {}

for ax_idx, method_name in enumerate(['mc_dropout', 'decoder_variance']):
    ax = axes[ax_idx]
    
    for variant, color in [(method_name, 'blue'), (f'{method_name}_ts', 'red')]:
        df = pd.read_csv(OUTPUT_DIR / f'eval_{variant}.csv')
        
        if len(df) > 0 and 'uncertainty' in df.columns:
            coverages, risks, auc = compute_risk_coverage(df, 'uncertainty')
            
            label = variant.replace('_', ' ').title()
            ax.plot(coverages, risks, label=f'{label} (AUC={auc:.3f})', color=color, linewidth=2)
            
            risk_coverage_results[variant] = {
                'coverages': coverages,
                'risks': risks,
                'auc': auc
            }
    
    ax.set_xlabel('Coverage', fontsize=12)
    ax.set_ylabel('Risk (1 - Accuracy)', fontsize=12)
    ax.set_title(f'Risk-Coverage: {method_name.replace("_", " ").title()}', fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'risk_coverage_curves.png', dpi=150, bbox_inches='tight')
print(f"Risk-coverage curves saved to: {OUTPUT_DIR / 'risk_coverage_curves.png'}")
plt.close()

# Save AUC
auc_summary = {k: v['auc'] for k, v in risk_coverage_results.items()}
with open(OUTPUT_DIR / 'risk_coverage_auc.json', 'w') as f:
    json.dump(auc_summary, f, indent=2)

print("\nRisk-Coverage AUC:")
for method, auc in auc_summary.items():
    print(f"  {method}: {auc:.4f}")

## 14. Final Summary and Report

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY - Method Comparison")
print("="*80)

# Cargar todas las métricas
det_metrics = json.load(open(OUTPUT_DIR / 'detection_metrics.json'))
cal_metrics = json.load(open(OUTPUT_DIR / 'calibration_metrics.json'))
temps = json.load(open(OUTPUT_DIR / 'temperatures.json'))
auc_summary = json.load(open(OUTPUT_DIR / 'risk_coverage_auc.json'))
uncertainty_auroc_data = json.load(open(OUTPUT_DIR / 'uncertainty_auroc.json'))

print("\n1. DETECTION METRICS (mAP@[0.5:0.95])")
print("-" * 80)
for method in ['baseline', 'baseline_ts', 'mc_dropout', 'mc_dropout_ts', 'decoder_variance', 'decoder_variance_ts']:
    mAP = det_metrics[method].get('mAP', 0.0)
    AP50 = det_metrics[method].get('AP50', 0.0)
    AP75 = det_metrics[method].get('AP75', 0.0)
    print(f"{method:25s}  mAP={mAP:.4f}  AP50={AP50:.4f}  AP75={AP75:.4f}")

print("\n2. CALIBRATION METRICS")
print("-" * 80)
print(f"{'Method':<25s} {'NLL ↓':>10s} {'Brier ↓':>10s} {'ECE ↓':>10s}")
print("-" * 80)
for method in ['baseline', 'baseline_ts', 'mc_dropout', 'mc_dropout_ts', 'decoder_variance', 'decoder_variance_ts']:
    nll = cal_metrics[method]['NLL']
    brier = cal_metrics[method]['Brier']
    ece = cal_metrics[method]['ECE']
    print(f"{method:<25s} {nll:>10.4f} {brier:>10.4f} {ece:>10.4f}")

print("\n3. OPTIMIZED TEMPERATURES")
print("-" * 80)
for method in ['baseline', 'mc_dropout', 'decoder_variance']:
    T = temps[method]['T']
    nll_before = temps[method]['nll_before']
    nll_after = temps[method]['nll_after']
    improvement = nll_before - nll_after
    print(f"{method:20s}  T={T:.4f}  NLL: {nll_before:.4f} → {nll_after:.4f} (Δ={improvement:.4f})")

print("\n4. RISK-COVERAGE AUC (lower is better)")
print("-" * 80)
for method, auc in auc_summary.items():
    print(f"{method:25s}  AUC={auc:.4f}")

print("\n5. UNCERTAINTY: AUROC TP vs FP (higher is better)")
print("-" * 80)
print(f"{'Method':<25s} {'AUROC ↑':>10s} {'Mean Unc TP':>15s} {'Mean Unc FP':>15s} {'Ratio':>10s}")
print("-" * 80)
for method, data in uncertainty_auroc_data.items():
    auroc = data['auroc']
    mean_tp = data['mean_unc_tp']
    mean_fp = data['mean_unc_fp']
    ratio = mean_fp / mean_tp if mean_tp > 0 else 0
    print(f"{method:<25s} {auroc:>10.4f} {mean_tp:>15.6f} {mean_fp:>15.6f} {ratio:>10.2f}x")

print("\n" + "="*80)
print("CONCLUSIONS")
print("="*80)
print("✓ Baseline: reference performance without uncertainty")
print("✓ Temperature Scaling: improves calibration without affecting mAP")
print("✓ MC-Dropout: provides epistemic uncertainty (K passes)")
print("✓ Decoder variance: uncertainty in single-pass (more efficient)")
print("✓ Methods+TS: better calibration maintaining detection")
print("✓ AUROC TP vs FP: validates that uncertainty detects errors")
print("="*80)

# Guardar reporte final
final_report = {
    'timestamp': datetime.now().isoformat(),
    'config': CONFIG,
    'detection_metrics': det_metrics,
    'calibration_metrics': cal_metrics,
    'temperatures': temps,
    'risk_coverage_auc': auc_summary,
    'uncertainty_auroc': uncertainty_auroc_data
}

with open(OUTPUT_DIR / 'final_report.json', 'w') as f:
    json.dump(final_report, f, indent=2)

print(f"\nFinal report saved to: {OUTPUT_DIR / 'final_report.json'}")
print(f"All artifacts in: {OUTPUT_DIR}")

## 15. Final Comparative Visualization

In [ ]:
fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)

methods = ['baseline', 'baseline_ts', 'mc_dropout', 'mc_dropout_ts', 'decoder_variance', 'decoder_variance_ts']
mAPs = [det_metrics[m].get('mAP', 0.0) for m in methods]
colors_map = ['lightblue', 'blue', 'lightcoral', 'red', 'lightgreen', 'green']

# 1. mAP Comparison
ax1 = fig.add_subplot(gs[0, :])
bars = ax1.bar(range(len(methods)), mAPs, color=colors_map, alpha=0.7)
ax1.set_xticks(range(len(methods)))
ax1.set_xticklabels([m.replace('_', '\n') for m in methods], fontsize=10)
ax1.set_ylabel('mAP@[0.5:0.95]', fontsize=12)
ax1.set_title('mAP Comparison Across Methods', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height, f'{mAPs[i]:.3f}', ha='center', va='bottom', fontsize=9)

# 2. Calibration Metrics
ax2 = fig.add_subplot(gs[1, 0])
nlls = [cal_metrics[m]['NLL'] for m in methods]
ax2.bar(range(len(methods)), nlls, color=colors_map, alpha=0.7)
ax2.set_xticks(range(len(methods)))
ax2.set_xticklabels([m.replace('_', '\n') for m in methods], fontsize=8)
ax2.set_ylabel('NLL ↓', fontsize=11)
ax2.set_title('Negative Log-Likelihood', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

ax3 = fig.add_subplot(gs[1, 1])
briers = [cal_metrics[m]['Brier'] for m in methods]
ax3.bar(range(len(methods)), briers, color=colors_map, alpha=0.7)
ax3.set_xticks(range(len(methods)))
ax3.set_xticklabels([m.replace('_', '\n') for m in methods], fontsize=8)
ax3.set_ylabel('Brier Score ↓', fontsize=11)
ax3.set_title('Brier Score', fontsize=12, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

ax4 = fig.add_subplot(gs[1, 2])
eces = [cal_metrics[m]['ECE'] for m in methods]
ax4.bar(range(len(methods)), eces, color=colors_map, alpha=0.7)
ax4.set_xticks(range(len(methods)))
ax4.set_xticklabels([m.replace('_', '\n') for m in methods], fontsize=8)
ax4.set_ylabel('ECE ↓', fontsize=11)
ax4.set_title('Expected Calibration Error', fontsize=12, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# 3. Temperature Scaling Effect
ax5 = fig.add_subplot(gs[2, 0])
base_methods = ['baseline', 'mc_dropout', 'decoder_variance']
Ts = [temps[m]['T'] for m in base_methods]
ax5.bar(range(len(base_methods)), Ts, color=['blue', 'red', 'green'], alpha=0.7)
ax5.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='T=1 (uncalibrated)')
ax5.set_xticks(range(len(base_methods)))
ax5.set_xticklabels([m.replace('_', '\n') for m in base_methods], fontsize=10)
ax5.set_ylabel('Temperature T', fontsize=11)
ax5.set_title('Optimal Temperatures', fontsize=12, fontweight='bold')
ax5.legend()
ax5.grid(axis='y', alpha=0.3)

# 4. Risk-Coverage AUC
ax6 = fig.add_subplot(gs[2, 1])
unc_methods = ['mc_dropout', 'mc_dropout_ts', 'decoder_variance', 'decoder_variance_ts']
aucs = [auc_summary.get(m, 0.0) for m in unc_methods]
colors_unc = ['lightcoral', 'red', 'lightgreen', 'green']
bars_auc = ax6.bar(range(len(unc_methods)), aucs, color=colors_unc, alpha=0.7)
ax6.set_xticks(range(len(unc_methods)))
ax6.set_xticklabels([m.replace('_', '\n') for m in unc_methods], fontsize=9)
ax6.set_ylabel('AUC (Risk-Coverage) ↓', fontsize=11)
ax6.set_title('Risk-Coverage AUC', fontsize=12, fontweight='bold')
ax6.grid(axis='y', alpha=0.3)
for i, bar in enumerate(bars_auc):
    height = bar.get_height()
    ax6.text(bar.get_x() + bar.get_width()/2., height, f'{aucs[i]:.3f}', ha='center', va='bottom', fontsize=8)

# 5. AUROC TP vs FP
ax7 = fig.add_subplot(gs[2, 2])
auroc_methods = list(uncertainty_auroc_data.keys())
aurocs = [uncertainty_auroc_data[m]['auroc'] for m in auroc_methods]
colors_auroc = ['lightcoral', 'red', 'lightgreen', 'green']
bars_auroc = ax7.bar(range(len(auroc_methods)), aurocs, color=colors_auroc, alpha=0.7)
ax7.axhline(y=0.5, color='black', linestyle='--', alpha=0.5, label='Random')
ax7.axhline(y=0.7, color='orange', linestyle='--', alpha=0.5, label='Good threshold')
ax7.set_xticks(range(len(auroc_methods)))
ax7.set_xticklabels([m.replace('_', '\n') for m in auroc_methods], fontsize=9)
ax7.set_ylabel('AUROC (FP detection) ↑', fontsize=11)
ax7.set_title('AUROC: Error Detection', fontsize=12, fontweight='bold')
ax7.legend(fontsize=8)
ax7.grid(axis='y', alpha=0.3)
ax7.set_ylim([0, 1])
for i, bar in enumerate(bars_auroc):
    height = bar.get_height()
    ax7.text(bar.get_x() + bar.get_width()/2., height, f'{aurocs[i]:.3f}', ha='center', va='bottom', fontsize=8)

# 6. Uncertainty ratio summary
ax8 = fig.add_subplot(gs[3, :])
ratios = [uncertainty_auroc_data[m]['mean_unc_fp'] / uncertainty_auroc_data[m]['mean_unc_tp'] 
          if uncertainty_auroc_data[m]['mean_unc_tp'] > 0 else 0 
          for m in auroc_methods]
bars_ratio = ax8.bar(range(len(auroc_methods)), ratios, color=colors_auroc, alpha=0.7)
ax8.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='Ratio = 1 (no difference)')
ax8.set_xticks(range(len(auroc_methods)))
ax8.set_xticklabels([m.replace('_', '\n') for m in auroc_methods], fontsize=10)
ax8.set_ylabel('Ratio Mean(Unc_FP) / Mean(Unc_TP)', fontsize=11)
ax8.set_title('Uncertainty Ratio: FP vs TP (>1 is desirable)', fontsize=12, fontweight='bold')
ax8.legend()
ax8.grid(axis='y', alpha=0.3)
for i, bar in enumerate(bars_ratio):
    height = bar.get_height()
    ax8.text(bar.get_x() + bar.get_width()/2., height, f'{ratios[i]:.2f}x', ha='center', va='bottom', fontsize=9)

plt.suptitle('Phase 5: Comprehensive Comparison of Uncertainty and Calibration Methods', 
             fontsize=16, fontweight='bold', y=0.997)

plt.savefig(OUTPUT_DIR / 'final_comparison_summary.png', dpi=150, bbox_inches='tight')
print(f"\nFinal visualization saved to: {OUTPUT_DIR / 'final_comparison_summary.png'}")
plt.close()

print("\n" + "="*80)
print("PHASE 5 COMPLETED")
print("="*80)
print(f"All results saved to: {OUTPUT_DIR}")
print("\nGenerated files:")
print("  - config.yaml")
print("  - temperatures.json")
print("  - detection_metrics.json")
print("  - calibration_metrics.json")
print("  - risk_coverage_auc.json")
print("  - uncertainty_auroc.json")
print("  - uncertainty_auroc_comparison.csv")
print("  - final_report.json")
print("  - detection_comparison.csv")
print("  - calibration_comparison.csv")
print("  - reliability_diagrams.png")
print("  - risk_coverage_curves.png")
print("  - uncertainty_analysis.png")
print("  - final_comparison_summary.png")
print("="*80)

In [ ]:
#!/usr/bin/env python3
"""
Phase 5 Optimization Verification Script
==========================================

This script verifies that:
1. Files from previous phases exist
2. Data formats are correct
3. Predictions are compatible
4. Estimates time savings
"""

import json
import sys
from pathlib import Path
from datetime import timedelta


# Colors for terminal
class Colors:
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    BLUE = "\033[94m"
    BOLD = "\033[1m"
    END = "\033[0m"


def check_file(path, description):
    """Verify if a file exists and return its info"""
    path = Path(path)
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{Colors.GREEN}✅ {description}{Colors.END}")
        print(f"   Location: {path}")
        print(f"   Size: {size_mb:.2f} MB")
        return True, size_mb
    else:
        print(f"{Colors.RED}❌ {description}{Colors.END}")
        print(f"   {Colors.YELLOW}Not found: {path}{Colors.END}")
        return False, 0


def verify_json_format(path, expected_keys):
    """Verify JSON has expected format"""
    try:
        with open(path, "r") as f:
            data = json.load(f)

        if not isinstance(data, list) or len(data) == 0:
            return False, "Not a list or empty"

        sample = data[0]
        missing_keys = [k for k in expected_keys if k not in sample]

        if missing_keys:
            return False, f"Missing keys: {missing_keys}"

        return True, f"{len(data)} records"
    except Exception as e:
        return False, str(e)


def main():
    print(f"\n{Colors.BOLD}{Colors.BLUE}{'='*70}")
    print("PHASE 5 OPTIMIZATION VERIFICATION")
    print(f"{'='*70}{Colors.END}\n")

    # Paths
    base_dir = Path("..")
    fase2_preds = base_dir / "fase 2" / "outputs" / "baseline" / "preds_raw.json"
    fase3_preds = (
        base_dir / "fase 3" / "outputs" / "mc_dropout" / "preds_mc_aggregated.json"
    )
    fase4_temp = (
        base_dir / "fase 4" / "outputs" / "temperature_scaling" / "temperature.json"
    )

    # Contadores
    files_found = 0
    total_files = 3
    time_saved = 0

    # ========================================================================
    print(f"{Colors.BOLD}1. PHASE 2 FILE VERIFICATION (Baseline){Colors.END}")
    print("-" * 70)

    exists, size = check_file(fase2_preds, "Baseline Predictions")
    if exists:
        files_found += 1
        time_saved += 45  # 45 minutes saved

        # Verificar formato
        valid, info = verify_json_format(
            fase2_preds, ["image_id", "category_id", "bbox", "score"]
        )
        if valid:
            print(f"   {Colors.GREEN}Format: ✅ Correct ({info}){Colors.END}")
        else:
            print(f"   {Colors.YELLOW}Format: ⚠️  {info}{Colors.END}")

    print()

    # ========================================================================
    print(f"{Colors.BOLD}2. PHASE 3 FILE VERIFICATION (MC-Dropout){Colors.END}")
    print("-" * 70)

    exists, size = check_file(fase3_preds, "MC-Dropout Predictions")
    if exists:
        files_found += 1
        time_saved += 90  # 90 minutes saved (K=5 is expensive)

        # Verificar formato
        valid, info = verify_json_format(
            fase3_preds, ["image_id", "category_id", "bbox", "score", "uncertainty"]
        )
        if valid:
            print(f"   {Colors.GREEN}Format: ✅ Correct ({info}){Colors.END}")
        else:
            print(f"   {Colors.YELLOW}Format: ⚠️  {info}{Colors.END}")

    print()

    # ========================================================================
    print(f"{Colors.BOLD}3. PHASE 4 FILE VERIFICATION (Temperature){Colors.END}")
    print("-" * 70)

    exists, size = check_file(fase4_temp, "Optimized Temperatures")
    if exists:
        files_found += 1
        time_saved += 2  # 2 minutes saved

        # Verificar formato
        try:
            with open(fase4_temp, "r") as f:
                temps = json.load(f)

            if "optimal_temperature" in temps:
                T = temps["optimal_temperature"]
                print(f"   {Colors.GREEN}Format: ✅ Correct (T={T:.4f}){Colors.END}")
            else:
                print(
                    f"   {Colors.YELLOW}Format: ⚠️  Missing 'optimal_temperature'{Colors.END}"
                )
        except Exception as e:
            print(f"   {Colors.YELLOW}Format: ⚠️  Error: {e}{Colors.END}")

    print()

    # ========================================================================
    print(f"{Colors.BOLD}{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}{Colors.END}")

    print(
        f"\n{Colors.BOLD}Files found:{Colors.END} {files_found}/{total_files}"
    )

    if files_found == total_files:
        print(f"{Colors.GREEN}✅ ALL files are available{Colors.END}")
    elif files_found > 0:
        print(f"{Colors.YELLOW}⚠️  Some files are available{Colors.END}")
    else:
        print(f"{Colors.RED}❌ NO files available{Colors.END}")

    # Estimación de tiempo
    print(f"\n{Colors.BOLD}Estimated time saved:{Colors.END}")

    if time_saved > 0:
        td = timedelta(minutes=time_saved)
        hours = td.seconds // 3600
        minutes = (td.seconds % 3600) // 60

        print(f"   {Colors.GREEN}⚡ ~{hours}h {minutes}min{Colors.END}")

        if files_found == total_files:
            print(f"\n{Colors.BOLD}Expected execution time:{Colors.END}")
            print(
                f"   {Colors.GREEN}📊 ~15-20 minutes{Colors.END} (only Decoder Variance)"
            )
        else:
            missing = total_files - files_found
            est_time = 137 - time_saved  # 137 min total original
            print(f"\n{Colors.BOLD}Expected execution time:{Colors.END}")
            print(
                f"   {Colors.YELLOW}📊 ~{est_time} minutes{Colors.END} (calculate {missing} missing method(s))"
            )
    else:
        print(f"   {Colors.RED}❌ 0 minutes{Colors.END}")
        print(f"\n{Colors.BOLD}Expected execution time:{Colors.END}")
        print(f"   {Colors.RED}📊 ~2 hours{Colors.END} (full inference)")

    # Recomendaciones
    print(f"\n{Colors.BOLD}{'='*70}")
    print("RECOMMENDATIONS")
    print(f"{'='*70}{Colors.END}")

    if files_found == total_files:
        print(
            f"{Colors.GREEN}✅ Perfect! You can run Phase 5 directly.{Colors.END}"
        )
        print(f"   The notebook will use all cached results.")
    elif files_found == 0:
        print(f"{Colors.YELLOW}⚠️  Run the following phases first:{Colors.END}")
        print(f"   1. Phase 2: Generate baseline predictions")
        print(f"   2. Phase 3: Generate MC-Dropout predictions")
        print(f"   3. Phase 4: Optimize temperatures")
        print(f"\n   Or run Phase 5 directly (will take ~2 hours)")
    else:
        print(f"{Colors.YELLOW}⚠️  You have partial optimization.{Colors.END}")

        if not (base_dir / fase2_preds).exists():
            print(f"   • Run Phase 2 for baseline predictions")
        if not (base_dir / fase3_preds).exists():
            print(f"   • Run Phase 3 for MC-Dropout predictions")
        if not (base_dir / fase4_temp).exists():
            print(f"   • Run Phase 4 for temperatures")

        print(f"\n   Or run Phase 5 now (will save ~{time_saved} min)")

    print(f"\n{Colors.BOLD}{'='*70}{Colors.END}\n")

    # Exit code
    return 0 if files_found == total_files else 1


if __name__ == "__main__":
    sys.exit(main())
